# ETL v2 — Extracción optimizada de tweets sobre fraude financiero (tier Enterprise 512 chars)

**Limitación del tier:** la API search/all en Enterprise estándar acepta queries de **máximo 512 caracteres**.
Por eso esta versión:

1. **En la query** solo van las exclusiones más críticas (política dura, deportes/k-pop). Ocupan ~115 chars y dejan ~400 chars libres para los positivos.
2. **En pandas** (post-API) está todo el resto del filtrado: metáfora, contexto monetario, longitud, ratios, marcas. No cuenta para el límite y permite ajustar sin gastar más llamadas.
3. **Diseño en 5 queries temáticas** (cada una entre 189 y 247 caracteres) que se unen y deduplican.

El CSV final se guarda en `../data/raw/` (no se versiona en Git).

⚠️ **PROTOCOLO DE PRUEBA RECOMENDADO** dada la limitación de presupuesto API:
- **Primera ejecución:** `PAGES_PER_QUERY = 1` → 500 tweets × 5 = máx. 2.500 brutos. Coste: 5 llamadas.
- Revisar muestra a ojo, ajustar filtros si hace falta.
- **Segunda ejecución (definitiva):** `PAGES_PER_QUERY = 8` → ~20.000 brutos. Coste: hasta 40 llamadas.


In [ ]:
# Imports y configuración
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

pd.set_option('display.max_colwidth', None)

load_dotenv()
TOKEN = os.getenv("X_BEARER_TOKEN", "")
if not TOKEN:
    raise ValueError(
        "Falta X_BEARER_TOKEN. Define la variable en un archivo .env "
        "(ver .env.example) o expórtala en tu shell antes de ejecutar el notebook."
    )

BASE_URL = "https://api.x.com/2/tweets/search/all"


In [ ]:
def search_all_posts_to_df(
    bearer_token: str,
    query: str,
    max_pages: int = 8,
    max_results: int = 500,
    sleep_seconds: float = 1.1,
    start_time: str = None,
    end_time: str = None,
    label: str = "",
) -> pd.DataFrame:
    """
    Descarga posts desde el archivo completo de X que matcheen `query`.
    Requiere acceso a search/all (Enterprise / Academic / DSA).
    `label` se añade como columna `query_label`.
    """
    headers = {"Authorization": f"Bearer {bearer_token}"}

    params = {
        "query": query,
        "max_results": max_results,
        "tweet.fields": ",".join([
            "id", "text", "created_at", "author_id", "lang", "geo",
            "public_metrics", "entities"
        ]),
        "expansions": ",".join(["author_id", "geo.place_id"]),
        "user.fields": ",".join([
            "id", "name", "username", "created_at", "location",
            "verified", "public_metrics"
        ]),
        "place.fields": ",".join([
            "id", "full_name", "country", "country_code", "place_type", "geo"
        ]),
        "sort_order": "recency",
    }

    if start_time:
        params["start_time"] = start_time
    if end_time:
        params["end_time"] = end_time

    all_rows = []
    next_token = None
    target_rows = max_pages * max_results

    for page in range(max_pages):
        if next_token:
            params["next_token"] = next_token
        else:
            params.pop("next_token", None)

        retries = 0
        while True:
            r = requests.get(BASE_URL, headers=headers, params=params, timeout=30)
            if r.status_code == 200:
                break
            elif r.status_code == 429:
                wait = min(60 * (2 ** retries), 900)
                print(f"  [Página {page}] Rate limit. Esperando {wait}s...")
                time.sleep(wait)
                retries += 1
                if retries > 5:
                    raise RuntimeError("Demasiados reintentos por rate limit.")
            elif r.status_code == 400:
                raise RuntimeError(
                    f"400 Bad Request: la query no es válida o es demasiado larga.\n"
                    f"Longitud: {len(query)} caracteres (límite Enterprise: 512).\n"
                    f"Detalle: {r.text}"
                )
            elif r.status_code == 403:
                raise RuntimeError(
                    f"403 Forbidden: tu plan no tiene acceso a search/all.\n"
                    f"Detalle: {r.text}"
                )
            else:
                raise RuntimeError(f"X API error {r.status_code}: {r.text}")

        payload = r.json()
        data = payload.get("data", []) or []
        includes = payload.get("includes", {}) or {}
        meta = payload.get("meta", {}) or {}

        users_by_id = {u["id"]: u for u in includes.get("users", []) or []}
        places_by_id = {p["id"]: p for p in includes.get("places", []) or []}

        for t in data:
            author = users_by_id.get(t.get("author_id"), {})
            place_id = (t.get("geo") or {}).get("place_id")
            place = places_by_id.get(place_id, {}) if place_id else {}
            entities = t.get("entities") or {}
            metrics = t.get("public_metrics") or {}
            author_metrics = author.get("public_metrics") or {}

            all_rows.append({
                "tweet_id": t.get("id"),
                "created_at": t.get("created_at"),
                "text": t.get("text"),
                "lang": t.get("lang"),
                "author_id": t.get("author_id"),
                "username": author.get("username"),
                "name": author.get("name"),
                "user_location": author.get("location"),
                "user_verified": author.get("verified"),
                "user_followers": author_metrics.get("followers_count"),
                "user_tweet_count": author_metrics.get("tweet_count"),
                "place_id": place_id,
                "place_full_name": place.get("full_name"),
                "place_country": place.get("country"),
                "place_country_code": place.get("country_code"),
                "place_type": place.get("place_type"),
                "retweet_count": metrics.get("retweet_count"),
                "reply_count": metrics.get("reply_count"),
                "like_count": metrics.get("like_count"),
                "quote_count": metrics.get("quote_count"),
                "n_hashtags": len(entities.get("hashtags", []) or []),
                "n_mentions": len(entities.get("mentions", []) or []),
                "n_urls": len(entities.get("urls", []) or []),
                "query_label": label,
            })

        print(f"  [{label} | Página {page + 1}/{max_pages}] Acumulados: {len(all_rows)} tweets")

        next_token = meta.get("next_token")
        if not next_token:
            print(f"  [{label}] No hay más páginas disponibles.")
            break
        if len(all_rows) >= target_rows:
            print(f"  [{label}] Objetivo de {target_rows} alcanzado.")
            break

        time.sleep(sleep_seconds)

    return pd.DataFrame(all_rows)


## Diseño minimalista de queries (compatibles con 512 chars)

Bloque común reducido al mínimo. El resto del filtrado se hace en pandas.


In [ ]:
# Exclusiones mínimas (las que más ruido cortan en EEUU)
EX_MIN = "-trump -biden -election -voter -ballot -nba -nfl -kpop"
GEO_OPS = "lang:en place_country:US -is:retweet -is:reply -is:nullcast"

def build_query(positive_core: str) -> str:
    """Une los positivos con el bloque común y verifica longitud."""
    q = f"({positive_core}) {EX_MIN} {GEO_OPS}"
    if len(q) > 512:
        raise ValueError(
            f"Query excede 512 chars ({len(q)}). Recórtala antes de enviar."
        )
    return q


QUERIES = {
    "phishing": build_query(
        '"phishing email" OR "phishing text" OR "scam text" '
        'OR "scam email" OR "smishing" OR "got a text from" OR "fake text"'
    ),
    "payment_apps": build_query(
        '"Zelle scam" OR "Venmo scam" OR "Cash App scam" '
        'OR "PayPal scam" OR "Apple Pay scam"'
    ),
    "crypto": build_query(
        '"rug pull" OR "pig butchering" OR "crypto scam" '
        'OR "bitcoin scam" OR "investment scam" OR "ponzi scheme"'
    ),
    "romance_monetary": build_query(
        '("romance scam" OR "dating scam") '
        '(money OR sent OR wire OR transferred)'
    ),
    "impersonation": build_query(
        '"IRS scam" OR "USPS scam" OR "FedEx scam" OR "toll scam" '
        'OR "fake delivery" OR "fake invoice" '
        'OR "tech support scam" OR "job scam"'
    ),
}

# Verificación
for label, q in QUERIES.items():
    print(f"  {label:<20} {len(q):>3} chars  ✓")
    print(f"    {q}\n")


## Configuración de la ejecución

⚠️ **MUY IMPORTANTE — leer antes de ejecutar**

Por defecto este notebook hace una **tirada DIAGNÓSTICA**: `PAGES_PER_QUERY = 1`.
- 5 llamadas a la API (una por query).
- Hasta 2.500 tweets brutos descargados.
- Sirve para revisar a ojo si la calidad es buena antes de gastar más presupuesto.

**Para la tirada definitiva:** cambia `PAGES_PER_QUERY = 8`.
- Hasta 40 llamadas (5 queries × 8 páginas).
- Hasta 20.000 tweets brutos.


In [ ]:
# 🚨 CAMBIA SOLO ESTE VALOR PARA TIRADA DEFINITIVA
PAGES_PER_QUERY = 1     # 1 = diagnóstica (~2.500 tweets) | 8 = definitiva (~20.000)

START = "2025-01-01T00:00:00Z"
END   = "2025-12-31T23:59:59Z"

# Estimación de coste
estimated_calls = len(QUERIES) * PAGES_PER_QUERY
estimated_max_tweets = estimated_calls * 500
print(f"Configuración:")
print(f"  Páginas por query:        {PAGES_PER_QUERY}")
print(f"  Llamadas API estimadas:   {estimated_calls}")
print(f"  Tweets máx. estimados:    {estimated_max_tweets}")
print(f"  Ventana temporal:         {START} → {END}")
print()
input_ok = input("Pulsa ENTER para ejecutar, o Ctrl+C para abortar... ")


## Ejecución de las 5 búsquedas

In [ ]:
dfs = []
for label, query in QUERIES.items():
    print(f"\n>>> Buscando: {label}")
    df_part = search_all_posts_to_df(
        bearer_token=TOKEN,
        query=query,
        max_pages=PAGES_PER_QUERY,
        max_results=500,
        sleep_seconds=1.1,
        start_time=START,
        end_time=END,
        label=label,
    )
    print(f">>> {label}: {len(df_part)} tweets")
    dfs.append(df_part)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\n=== TOTAL BRUTO (con duplicados entre queries): {len(df_raw)} ===")


## Deduplicación entre queries

Un tweet puede haber sido capturado por varias queries. Lo dejamos una vez y guardamos en `query_labels` todas las categorías que lo identificaron — eso es información útil para el clasificador posterior.


In [ ]:
df_raw["query_labels"] = df_raw.groupby("tweet_id")["query_label"].transform(
    lambda s: ",".join(sorted(set(s)))
)
df_dedup = df_raw.drop_duplicates(subset="tweet_id", keep="first").reset_index(drop=True)
df_dedup = df_dedup.drop(columns=["query_label"])

print(f"Tras deduplicar: {len(df_dedup)} tweets únicos")
print(f"Multi-categoría: {(df_dedup['query_labels'].str.contains(',')).sum()}")


## Filtros post-API en pandas (la pieza CLAVE de esta versión)

Aquí está la mayor parte del filtrado de ruido, ya que las exclusiones en la query están muy limitadas por los 512 chars.

| Filtro | Qué corta |
|---|---|
| Patrones de metáfora (`"life is a scam"`, etc.) | Discurso figurado/abstracto |
| Política blanda (palabras adicionales en el texto) | Tweets políticos que escaparon |
| Cuentas-marca / news | Spam de afiliados y prensa |
| Longitud mínima (clean ≥ 40) | Tweets demasiado cortos |
| Ratio hashtags / menciones | Keyword stuffing |
| Contexto monetario obligatorio | Discurso sin componente financiero real |
| Geolocalización suave | Usuarios no-EEUU que escaparon de `place_country:US` |


In [ ]:
import re

# Patrones de descarte: metáfora, política suave, marcas
METAPHOR_PATTERNS = re.compile(
    r"\b(life is a scam|love is a scam|system is a scam|"
    r"capitalism is a scam|college is a scam|dating is a scam|"
    r"my life is a scam|everything is a scam|"
    r"life'?s a scam|love'?s a scam)\b",
    re.IGNORECASE,
)

SOFT_POLITICS_PATTERNS = re.compile(
    r"\b(harris|desantis|newsom|kamala|obama|congress|senate|senator|"
    r"republican|democrat|gop|maga|lawmaker|politician|"
    r"voter fraud|election fraud|stolen election|rigged election|"
    r"deep state)\b",
    re.IGNORECASE,
)

BRAND_USERNAMES = {
    "lifelock", "aura", "norton", "nortononline", "mcafee",
    "cnn", "foxnews", "msnbc", "nytimes", "wsj", "reuters",
    "ap", "bbcworld", "abc", "abcnews", "nbcnews", "cbsnews",
}

# Patrón de catfishing emocional sin componente monetario
EMOTIONAL_ONLY = re.compile(
    r"\b(catfish|catfished|catfishing|emotional scam|"
    r"scammed my heart|broke my heart)\b",
    re.IGNORECASE,
)

# Términos monetarios (al menos uno debe aparecer)
MONEY_TERMS = re.compile(
    r"(\b(money|cash|dollar|dollars|usd|paid|payment|sent|wired|wire|"
    r"transfer|refund|bank|account|credit\s*card|debit|wallet|invoice|"
    r"charged|stolen|lost|victim|scammed|defrauded|deposit|withdraw|"
    r"check|venmo|zelle|paypal|crypto|bitcoin|coinbase)\b|"
    r"\$\s*\d+|\d+\s*(k|dollars|usd))",
    re.IGNORECASE,
)

URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")

def clean_for_length(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

us_keywords = [
    "united states", "usa", "u.s.", " us ", " us,", "america",
    "new york", "california", "texas", "florida", "chicago",
    "los angeles", "boston", "seattle", "miami", "atlanta",
    "dallas", "houston", "philadelphia", "phoenix", "denver",
    "washington", "san francisco", "san diego",
]

def looks_us(row):
    if row.get("place_country_code") == "US":
        return True
    loc = " " + str(row.get("user_location") or "").lower() + " "
    return any(k in loc for k in us_keywords)

# Construir columnas auxiliares
df = df_dedup.copy()
df["clean_text"] = df["text"].apply(clean_for_length)
df["clean_len"] = df["clean_text"].str.len()
df["n_words"] = df["clean_text"].str.split().str.len().fillna(0).astype(int)
df["hashtag_ratio"] = (df["n_hashtags"] / df["n_words"].replace(0, 1)).fillna(0)
df["mention_ratio"] = (df["n_mentions"] / df["n_words"].replace(0, 1)).fillna(0)

df["is_metaphor"]      = df["text"].fillna("").apply(lambda t: bool(METAPHOR_PATTERNS.search(t)))
df["is_soft_politics"] = df["text"].fillna("").apply(lambda t: bool(SOFT_POLITICS_PATTERNS.search(t)))
df["is_emotional"]     = df["text"].fillna("").apply(lambda t: bool(EMOTIONAL_ONLY.search(t)))
df["has_money"]        = df["text"].fillna("").apply(lambda t: bool(MONEY_TERMS.search(t)))
df["is_brand_account"] = df["username"].fillna("").str.lower().isin(BRAND_USERNAMES)
df["likely_us"]        = df.apply(looks_us, axis=1)

print("Distribución de descartes potenciales (sobre %d tweets):" % len(df))
print(f"  Metáfora / figurado:     {df['is_metaphor'].sum()} ({df['is_metaphor'].mean()*100:.1f}%)")
print(f"  Política suave:          {df['is_soft_politics'].sum()} ({df['is_soft_politics'].mean()*100:.1f}%)")
print(f"  Solo emocional:          {df['is_emotional'].sum()} ({df['is_emotional'].mean()*100:.1f}%)")
print(f"  SIN contexto monetario:  {(~df['has_money']).sum()} ({(~df['has_money']).mean()*100:.1f}%)")
print(f"  Cuentas-marca:           {df['is_brand_account'].sum()} ({df['is_brand_account'].mean()*100:.1f}%)")
print(f"  Probablemente US:        {df['likely_us'].sum()} ({df['likely_us'].mean()*100:.1f}%)")


In [ ]:
# Aplicar los filtros
mask = (
    (df["clean_len"] >= 40) &
    (df["has_money"]) &
    (df["likely_us"]) &
    (df["hashtag_ratio"] < 0.3) &
    (df["mention_ratio"] < 0.4) &
    (~df["is_metaphor"]) &
    (~df["is_soft_politics"]) &
    (~df["is_emotional"]) &
    (~df["is_brand_account"])
)

df_clean = df[mask].reset_index(drop=True)

print("=== RESUMEN DE LIMPIEZA ===")
print(f"Brutos descargados:       {len(df_raw)}")
print(f"Únicos tras deduplicar:   {len(df_dedup)}")
print(f"Limpios finales:          {len(df_clean)}")
print(f"Tasa de retención:        {len(df_clean) / max(len(df_dedup), 1) * 100:.1f}%")
print()
print("Distribución por categoría temática:")
for label in QUERIES.keys():
    n = df_clean["query_labels"].str.contains(label).sum()
    print(f"  {label:<20} {n:>6}")


## Guardado de resultados

In [ ]:
os.makedirs("../data/raw", exist_ok=True)

df_dedup.to_csv("../data/raw/scam_us_raw_dedup.csv", index=False, encoding="utf-8")
df_clean.to_csv("../data/raw/scam_us_clean.csv", index=False, encoding="utf-8")

print("Guardados:")
print(f"  ../data/raw/scam_us_raw_dedup.csv  ({len(df_dedup)} filas)")
print(f"  ../data/raw/scam_us_clean.csv      ({len(df_clean)} filas)")
print()
print("⚠️  El CSV NO se sube a GitHub (está en .gitignore).")
print("⚠️  Haz copia de seguridad en Drive/Dropbox cuanto antes para no perder el corpus.")


## Inspección a ojo (muy importante en la tirada diagnóstica)

Revisa esta muestra antes de hacer la tirada grande. Si ves muchos tweets que claramente no deberían estar, dímelo y ajustamos filtros antes de gastar más presupuesto.


In [ ]:
# Muestra aleatoria para revisión visual
print("=== MUESTRA ALEATORIA DE 20 TWEETS LIMPIOS ===\n")
sample = df_clean.sample(min(20, len(df_clean)), random_state=42)
for _, row in sample.iterrows():
    print(f"[{row['query_labels']}] @{row['username']} — {row['user_location']}")
    print(f"  {row['text'][:280]}")
    print()


In [ ]:
# Inspección de qué se descartó y por qué (útil para diagnosticar filtros demasiado estrictos)
print("=== MUESTRA DE TWEETS DESCARTADOS POR FALTA DE CONTEXTO MONETARIO ===")
discarded_no_money = df[~df["has_money"]].sample(min(5, (~df["has_money"]).sum()), random_state=42)
for _, row in discarded_no_money.iterrows():
    print(f"  - {row['text'][:200]}")

print("\n=== MUESTRA DE TWEETS DESCARTADOS POR METÁFORA ===")
discarded_metaphor = df[df["is_metaphor"]].sample(min(5, df["is_metaphor"].sum()), random_state=42)
for _, row in discarded_metaphor.iterrows():
    print(f"  - {row['text'][:200]}")
